## 1. Membership Functions and Fuzzy Ranges
This cell defines the triangular membership function used throughout the notebook and declares the fuzzy sets for rainfall, temperature, soil moisture, soil nutrients, and crop yield. It is the foundation of the fuzzy inference system, because every later cell depends on these ranges and helper rules.

In [40]:
from pprint import pprint

def triangular_membership(x: float, a: float, b: float, c: float) -> float:
    if a == b:
        if x <= b:
            return 1.0
        if x >= c:
            return 0.0
        return (c - x) / (c - b)

    if b == c:
        if x <= a:
            return 0.0
        if x >= b:
            return 1.0
        return (x - a) / (b - a)

    if x <= a or x >= c:
        return 0.0
    if x == b:
        return 1.0
    if x < b:
        return (x - a) / (b - a)
    return (c - x) / (c - b)


membership_ranges = {
    "rainfall": {
        "low": (0, 0, 50),
        "moderate": (25, 50, 75),
        "high": (50, 100, 100),
    },
    "temperature": {
        "cold": (10, 10, 27),
        "warm": (18, 27, 34),
        "hot": (27, 40, 40),
    },
    "soil_moisture": {
        "dry": (0, 0, 50),
        "moist": (25, 50, 75),
        "wet": (50, 100, 100),
    },
    "soil_nutrients": {
        "poor": (0, 0, 50),
        "fair": (25, 50, 75),
        "rich": (50, 100, 100),
    },
    "yield": {
        "very_low": (0, 0, 1.5),
        "low": (0.75, 1.5, 3.0),
        "medium": (1.5, 3.0, 4.5),
        "high": (3.0, 3.4, 4.6),
        "very_high": (3.4, 4.6, 6.0),
    },
}

## 2. Input Fuzzification Helpers
This cell converts crisp input values into fuzzy membership scores. In other words, it translates raw measurements into degrees such as `low`, `moderate`, `high`, `dry`, `moist`, or `rich`, which the rule engine can then evaluate.

In [41]:
def fuzzify_value(x: float, variable_name: str) -> dict:
    terms = membership_ranges[variable_name]
    memberships = {}

    for term_name, (a, b, c) in terms.items():
        memberships[term_name] = triangular_membership(x, a, b, c)

    return memberships


def fuzzify_inputs(rainfall: float, temperature: float, soil_moisture: float, soil_nutrients: float) -> dict:
    return {
        "rainfall": fuzzify_value(rainfall, "rainfall"),
        "temperature": fuzzify_value(temperature, "temperature"),
        "soil_moisture": fuzzify_value(soil_moisture, "soil_moisture"),
        "soil_nutrients": fuzzify_value(soil_nutrients, "soil_nutrients"),
    }

## 3. Sample Test Cases
This cell stores example scenarios used to test the fuzzy model. Each case includes environmental inputs and an expected yield label so developers can compare the model's prediction against a target outcome.

In [42]:
sample_cases = [
    {"case_id": 1, "category": "Happy Path", "rainfall": 100, "temperature": 28, "soil_moisture": 65, "soil_nutrients": 60, "expected_yield": "High"},
    {"case_id": 2, "category": "Happy Path", "rainfall": 120, "temperature": 26, "soil_moisture": 70, "soil_nutrients": 58, "expected_yield": "High"},
    {"case_id": 3, "category": "Happy Path", "rainfall": 150, "temperature": 27, "soil_moisture": 68, "soil_nutrients": 62, "expected_yield": "High"},
    {"case_id": 4, "category": "Happy Path", "rainfall": 180, "temperature": 25, "soil_moisture": 72, "soil_nutrients": 55, "expected_yield": "High"},
    {"case_id": 5, "category": "Happy Path", "rainfall": 200, "temperature": 29, "soil_moisture": 66, "soil_nutrients": 88, "expected_yield": "Very High"},
    {"case_id": 6, "category": "Happy Path", "rainfall": 220, "temperature": 24, "soil_moisture": 75, "soil_nutrients": 90, "expected_yield": "Very High"},
    {"case_id": 7, "category": "Happy Path", "rainfall": 250, "temperature": 26, "soil_moisture": 70, "soil_nutrients": 85, "expected_yield": "Very High"},
    {"case_id": 8, "category": "Edge Cases", "rainfall": 0, "temperature": 45, "soil_moisture": 5, "soil_nutrients": 20, "expected_yield": "Very Low"},
    {"case_id": 9, "category": "Edge Cases", "rainfall": 10, "temperature": 43, "soil_moisture": 8, "soil_nutrients": 22, "expected_yield": "Very Low"},
    {"case_id": 10, "category": "Edge Cases", "rainfall": 5, "temperature": 44, "soil_moisture": 6, "soil_nutrients": 18, "expected_yield": "Very Low"},
    {"case_id": 11, "category": "Edge Cases", "rainfall": 300, "temperature": 25, "soil_moisture": 100, "soil_nutrients": 25, "expected_yield": "Very Low"},
    {"case_id": 12, "category": "Edge Cases", "rainfall": 280, "temperature": 24, "soil_moisture": 98, "soil_nutrients": 40, "expected_yield": "Low"},
    {"case_id": 13, "category": "Edge Cases", "rainfall": 320, "temperature": 26, "soil_moisture": 100, "soil_nutrients": 15, "expected_yield": "Very Low"},
    {"case_id": 14, "category": "Edge Cases", "rainfall": 15, "temperature": 42, "soil_moisture": 10, "soil_nutrients": 20, "expected_yield": "Very Low"},
    {"case_id": 15, "category": "Stress Test", "rainfall": 150, "temperature": 40, "soil_moisture": 80, "soil_nutrients": 65, "expected_yield": "Medium"},
    {"case_id": 16, "category": "Stress Test", "rainfall": 160, "temperature": 39, "soil_moisture": 78, "soil_nutrients": 60, "expected_yield": "Medium"},
    {"case_id": 17, "category": "Stress Test", "rainfall": 140, "temperature": 41, "soil_moisture": 82, "soil_nutrients": 68, "expected_yield": "Medium"},
    {"case_id": 18, "category": "Stress Test", "rainfall": 170, "temperature": 38, "soil_moisture": 75, "soil_nutrients": 55, "expected_yield": "Medium"},
    {"case_id": 19, "category": "Stress Test", "rainfall": 130, "temperature": 42, "soil_moisture": 85, "soil_nutrients": 35, "expected_yield": "Low"},
    {"case_id": 20, "category": "Stress Test", "rainfall": 155, "temperature": 37, "soil_moisture": 77, "soil_nutrients": 62, "expected_yield": "Medium"},
]

## 4. Fuzzy Rule Base
This cell defines the complete fuzzy rule table. Each rule connects a combination of rainfall, temperature, soil moisture, and soil nutrient conditions to an output yield category, which is the core decision logic of the system.

In [43]:
rule_rows = [
    (1, "low", "cold", "dry", "poor", "very_low"),
    (2, "low", "cold", "dry", "fair", "very_low"),
    (3, "low", "cold", "dry", "rich", "low"),
    (4, "low", "cold", "moist", "poor", "very_low"),
    (5, "low", "cold", "moist", "fair", "low"),
    (6, "low", "cold", "moist", "rich", "low"),
    (7, "low", "cold", "wet", "poor", "low"),
    (8, "low", "cold", "wet", "fair", "low"),
    (9, "low", "cold", "wet", "rich", "medium"),
    (10, "low", "warm", "dry", "poor", "very_low"),
    (11, "low", "warm", "dry", "fair", "low"),
    (12, "low", "warm", "dry", "rich", "low"),
    (13, "low", "warm", "moist", "poor", "low"),
    (14, "low", "warm", "moist", "fair", "low"),
    (15, "low", "warm", "moist", "rich", "medium"),
    (16, "low", "warm", "wet", "poor", "low"),
    (17, "low", "warm", "wet", "fair", "medium"),
    (18, "low", "warm", "wet", "rich", "medium"),
    (19, "low", "hot", "dry", "poor", "very_low"),
    (20, "low", "hot", "dry", "fair", "very_low"),
    (21, "low", "hot", "dry", "rich", "low"),
    (22, "low", "hot", "moist", "poor", "very_low"),
    (23, "low", "hot", "moist", "fair", "low"),
    (24, "low", "hot", "moist", "rich", "low"),
    (25, "low", "hot", "wet", "poor", "low"),
    (26, "low", "hot", "wet", "fair", "low"),
    (27, "low", "hot", "wet", "rich", "medium"),
    (28, "moderate", "cold", "dry", "poor", "low"),
    (29, "moderate", "cold", "dry", "fair", "low"),
    (30, "moderate", "cold", "dry", "rich", "medium"),
    (31, "moderate", "cold", "moist", "poor", "low"),
    (32, "moderate", "cold", "moist", "fair", "medium"),
    (33, "moderate", "cold", "moist", "rich", "medium"),
    (34, "moderate", "cold", "wet", "poor", "low"),
    (35, "moderate", "cold", "wet", "fair", "medium"),
    (36, "moderate", "cold", "wet", "rich", "medium"),
    (37, "moderate", "warm", "dry", "poor", "low"),
    (38, "moderate", "warm", "dry", "fair", "medium"),
    (39, "moderate", "warm", "dry", "rich", "medium"),
    (40, "moderate", "warm", "moist", "poor", "medium"),
    (41, "moderate", "warm", "moist", "fair", "medium"),
    (42, "moderate", "warm", "moist", "rich", "high"),
    (43, "moderate", "warm", "wet", "poor", "medium"),
    (44, "moderate", "warm", "wet", "fair", "high"),
    (45, "moderate", "warm", "wet", "rich", "high"),
    (46, "moderate", "hot", "dry", "poor", "low"),
    (47, "moderate", "hot", "dry", "fair", "low"),
    (48, "moderate", "hot", "dry", "rich", "medium"),
    (49, "moderate", "hot", "moist", "poor", "low"),
    (50, "moderate", "hot", "moist", "fair", "medium"),
    (51, "moderate", "hot", "moist", "rich", "medium"),
    (52, "moderate", "hot", "wet", "poor", "medium"),
    (53, "moderate", "hot", "wet", "fair", "medium"),
    (54, "moderate", "hot", "wet", "rich", "high"),
    (55, "high", "cold", "dry", "poor", "low"),
    (56, "high", "cold", "dry", "fair", "medium"),
    (57, "high", "cold", "dry", "rich", "medium"),
    (58, "high", "cold", "moist", "poor", "medium"),
    (59, "high", "cold", "moist", "fair", "medium"),
    (60, "high", "cold", "moist", "rich", "high"),
    (61, "high", "cold", "wet", "poor", "very_low"),
    (62, "high", "cold", "wet", "fair", "medium"),
    (63, "high", "cold", "wet", "rich", "medium"),
    (64, "high", "warm", "dry", "poor", "medium"),
    (65, "high", "warm", "dry", "fair", "medium"),
    (66, "high", "warm", "dry", "rich", "high"),
    (67, "high", "warm", "moist", "poor", "medium"),
    (68, "high", "warm", "moist", "fair", "high"),
    (69, "high", "warm", "moist", "rich", "very_high"),
    (70, "high", "warm", "wet", "poor", "very_low"),
    (71, "high", "warm", "wet", "fair", "high"),
    (72, "high", "warm", "wet", "rich", "very_high"),
    (73, "high", "hot", "dry", "poor", "low"),
    (74, "high", "hot", "dry", "fair", "medium"),
    (75, "high", "hot", "dry", "rich", "medium"),
    (76, "high", "hot", "moist", "poor", "medium"),
    (77, "high", "hot", "moist", "fair", "medium"),
    (78, "high", "hot", "moist", "rich", "high"),
    (79, "high", "hot", "wet", "poor", "low"),
    (80, "high", "hot", "wet", "fair", "medium"),
    (81, "high", "hot", "wet", "rich", "medium"),
]

rules = [
    {
        "id": rule_id,
        "rainfall": rainfall,
        "temperature": temperature,
        "soil_moisture": soil_moisture,
        "soil_nutrients": soil_nutrients,
        "yield": yield_label,
    }
    for rule_id, rainfall, temperature, soil_moisture, soil_nutrients, yield_label in rule_rows
]

## 5. Rule Evaluation
This cell evaluates all fuzzy rules against the fuzzified inputs. It computes which rules fire, how strongly they fire, and aggregates the resulting output activations for the yield categories.

In [44]:
def evaluate_rules(fuzzy_inputs: dict, rules: list) -> tuple[dict, list]:
    output_activations = {
        "very_low": 0.0,
        "low": 0.0,
        "medium": 0.0,
        "high": 0.0,
        "very_high": 0.0,
    }

    fired_rules = []

    for rule in rules:
        firing_strength = min(
            fuzzy_inputs["rainfall"][rule["rainfall"]],
            fuzzy_inputs["temperature"][rule["temperature"]],
            fuzzy_inputs["soil_moisture"][rule["soil_moisture"]],
            fuzzy_inputs["soil_nutrients"][rule["soil_nutrients"]],
        )

        if firing_strength > 0:
            fired_rules.append({
                "id": rule["id"],
                "conditions": {
                    "rainfall": rule["rainfall"],
                    "temperature": rule["temperature"],
                    "soil_moisture": rule["soil_moisture"],
                    "soil_nutrients": rule["soil_nutrients"],
                },
                "yield": rule["yield"],
                "firing_strength": firing_strength,
            })

        output_activations[rule["yield"]] = max(
            output_activations[rule["yield"]],
            firing_strength
        )

    return output_activations, fired_rules

## 6. Output Aggregation and Defuzzification
This cell turns the fuzzy output activations into a single crisp yield value using centroid defuzzification. It also maps that numeric result back into the closest human-readable yield label.

In [45]:
def aggregated_output_membership(y_value: float, output_activations: dict) -> float:
    clipped_memberships = []

    for label, activation in output_activations.items():
        membership = triangular_membership(y_value, *membership_ranges["yield"][label])
        clipped_memberships.append(min(activation, membership))

    return max(clipped_memberships)


def defuzzify(output_activations: dict, start: float = 0.0, end: float = 6.0, step: float = 0.01) -> float:
    sample_points = []
    current = start

    while current <= end + 1e-9:
        sample_points.append(round(current, 4))
        current += step

    memberships = [aggregated_output_membership(point, output_activations) for point in sample_points]
    denominator = sum(memberships)

    if denominator == 0:
        return 0.0

    numerator = sum(point * membership for point, membership in zip(sample_points, memberships))
    return numerator / denominator


def label_from_crisp_yield(crisp_yield: float) -> str:
    memberships = {
        label: triangular_membership(crisp_yield, *membership_ranges["yield"][label])
        for label in membership_ranges["yield"]
    }
    best_label = max(memberships, key=memberships.get)
    return best_label.replace("_", " ").title()

## 7. Prediction Run and Output Summary
This cell runs the full prediction pipeline for every sample case. It prints the predicted yield value, predicted label, and the strongest contributing rule, then summarizes how many of the 20 test cases matched the expected yield labels and reports the overall accuracy percentage.

In [46]:
def predict_case(row: dict) -> dict:
    fuzzy_inputs = fuzzify_inputs(
        rainfall=row["rainfall"],
        temperature=row["temperature"],
        soil_moisture=row["soil_moisture"],
        soil_nutrients=row["soil_nutrients"],
    )
    output_activations, fired_rules = evaluate_rules(fuzzy_inputs, rules)
    crisp_yield = defuzzify(output_activations)
    strongest_rule = max(fired_rules, key=lambda rule: rule["firing_strength"], default=None)

    return {
        "predicted_yield_value": round(crisp_yield, 2),
        "predicted_yield_label": label_from_crisp_yield(crisp_yield),
        "top_rule_id": strongest_rule["id"] if strongest_rule else None,
        "top_rule_strength": round(strongest_rule["firing_strength"], 3) if strongest_rule else None,
    }


prediction_rows = []

for row in sample_cases:
    prediction_rows.append({**row, **predict_case(row)})

print("Case predictions:")
for row in prediction_rows:
    print(
        f"Case {row['case_id']:>2}: expected={row['expected_yield']:<9} predicted={row['predicted_yield_label']:<9} value={row['predicted_yield_value']:.2f} top_rule={row['top_rule_id']} strength={row['top_rule_strength']}"
    )

matched_cases = sum(
    1 for row in prediction_rows
    if row["expected_yield"] == row["predicted_yield_label"]
)
total_cases = len(prediction_rows)
mismatched_cases = total_cases - matched_cases
accuracy = matched_cases / total_cases if total_cases else 0.0

print("\nAccuracy summary:")
print(f"Matched cases   : {matched_cases}/{total_cases}")
print(f"Mismatched cases: {mismatched_cases}/{total_cases}")
print(f"Accuracy        : {accuracy:.1%}")

Case predictions:
Case  1: expected=High      predicted=High      value=3.99 top_rule=68 strength=0.4
Case  2: expected=High      predicted=High      value=3.97 top_rule=71 strength=0.4
Case  3: expected=High      predicted=Very High value=4.31 top_rule=71 strength=0.36
Case  4: expected=High      predicted=High      value=3.69 top_rule=71 strength=0.44
Case  5: expected=Very High predicted=Very High value=4.10 top_rule=69 strength=0.36
Case  6: expected=Very High predicted=Very High value=4.14 top_rule=72 strength=0.5
Case  7: expected=Very High predicted=Very High value=4.42 top_rule=72 strength=0.4
Case  8: expected=Very Low  predicted=Very Low  value=0.55 top_rule=19 strength=0.6
Case  9: expected=Very Low  predicted=Very Low  value=0.56 top_rule=19 strength=0.56
Case 10: expected=Very Low  predicted=Very Low  value=0.54 top_rule=19 strength=0.64
Case 11: expected=Very Low  predicted=Very Low  value=0.58 top_rule=70 strength=0.5
Case 12: expected=Low       predicted=Medium    value